In [1]:
import pandas as pd
import pickle as pkl
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV
from sklearn import metrics
from sklearn.pipeline import Pipeline

In [2]:
neg_to_pos_ratio = 1
# seed = 42

study_design = 'is_guideline'

filename_pos_human = 'pubmed_articles_pos_human.csv'

filename_neg_human = 'pubmed_articles_neg_human.csv'

In [3]:
with open('pos_neg_dataset_ids.pkl', 'rb') as f:
    prop_typ_id_dict= pkl.load(f)

In [8]:
for key in prop_typ_id_dict.keys():
    print(key)
    print('pos:', len(prop_typ_id_dict[key]['pos']))
    print('neg:', len(prop_typ_id_dict[key]['neg']))
    print()

is_rct
pos: 489295
neg: 5585042

is_sr
pos: 121063
neg: 5659292

is_casecontrol
pos: 65790
neg: 5628219

is_ct
pos: 593277
neg: 5564942

is_cohort
pos: 78688
neg: 5632460

is_guideline
pos: 33826
neg: 5676411

is_consensus
pos: 10093
neg: 5678494

is_qual
pos: 7577
neg: 5673091

is_ctprotocol
pos: 3235
neg: 5682702



In [7]:
prop_typ_id_dict

{'is_rct': {'pos': [30,
   34,
   37,
   59,
   98,
   103,
   193,
   230,
   238,
   265,
   275,
   297,
   324,
   326,
   328,
   332,
   361,
   378,
   391,
   396,
   436,
   440,
   578,
   579,
   595,
   675,
   731,
   767,
   820,
   822,
   870,
   871,
   885,
   896,
   898,
   951,
   959,
   975,
   989,
   990,
   992,
   993,
   1019,
   1047,
   1049,
   1064,
   1119,
   1122,
   1135,
   1173,
   1193,
   1209,
   1211,
   1212,
   1217,
   1232,
   1238,
   1240,
   1243,
   1266,
   1272,
   1274,
   1346,
   1349,
   1350,
   1357,
   1436,
   1440,
   1478,
   1482,
   1538,
   1541,
   1576,
   1580,
   1631,
   1636,
   1864,
   1886,
   1956,
   1971,
   1973,
   1982,
   1983,
   1984,
   1985,
   1986,
   2018,
   2066,
   2069,
   2070,
   2071,
   2072,
   2073,
   2075,
   2076,
   2079,
   2080,
   2081,
   2084,
   2085,
   2088,
   2109,
   2116,
   2124,
   2127,
   2221,
   2225,
   2239,
   2240,
   2257,
   2258,
   2317,
   2318,
   2319,
   2

In [3]:
print('loading positive data')
pos_data_human = pd.read_csv(
         filename_pos_human,
)




loading positive data


In [4]:
prop_typ_id_dict.keys()

dict_keys(['is_rct', 'is_sr', 'is_casecontrol', 'is_ct', 'is_cohort', 'is_guideline', 'is_consensus', 'is_qual', 'is_ctprotocol'])

In [6]:
print('setting positive rows') 
pos_ids = prop_typ_id_dict[study_design]['pos']
pos_set  = pos_data_human.loc[pos_ids]
del pos_data_human

setting positive rows


In [17]:
len(pos_set['abstract_plaintext'])

33826

In [18]:
pos_set['abstract_plaintext'].count()

17015

In [19]:
len(pos_set['abstract_plaintext']) - pos_set['abstract_plaintext'].count()

16811

In [24]:
notna = lambda df: [not x for x in df.isna()]

In [26]:
pos_notna  = pos_set[ notna(pos_set['abstract_plaintext'])]

In [42]:
pos_notna['title'][8583755]

'[Diagnosis and treatment with steroids for patients with Duchenne muscular dystrophy: experience and recommendations for Mexico. Administración del Patrimonio de la Beneficencia Pública. Asociación de Distrofia Muscular de Occidente].'

In [41]:
pos_notna['abstract_plaintext'][8583755]

'Duchenne muscular dystrophy is a severe, debilitating and progressive disease that affects 1 in 3,500 live male births in the world. The diagnosis should be confirmed by genetic testing to identify the mutation in the DMD gene or muscle biopsy and immunostaining to demonstrate the absence of dystrophin. Although up to now continues to be an incurable disease, this does not mean it has no treatment. Treatment should be multidisciplinary, looking for the functionality of the patient and avoiding or correcting complications, mainly cardio-respiratory and skeletal. Many proposals have been evaluated and implemented with the aim of improving the quality of life for these patients. The long-term steroids have shown significant benefits, such as prolonging ambulation, reduce the need for spinal surgery, improve cardiorespiratory function and increase survival and the quality of life. This document presents the recommendations based on the experience of the working group and experts worldwide

In [33]:
pos_notna['abstract_plaintext']

383        Radionuclide imaging of cardiac function repre...
1446       The primary treatment of a melanoma is surgica...
3687       --The practice guideline 'Thyroid disorders' d...
4044       Asthma is a common, potentially serious medica...
4296       BACKGROUND\n\n\nThis program examined the impa...
                                 ...                        
8583328    The transition period from child to adult repr...
8583741    Although traumatic brain injury (TBI) can happ...
8583755    Duchenne muscular dystrophy is a severe, debil...
8584272    The recommendations for the practical stabilit...
8584466    CONTEXT\n\n\nThe most recent summary of the Eu...
Name: abstract_plaintext, Length: 17015, dtype: object